# 10-Year Treasury Yield Full Refresh Pipeline

## Purpose
Full-refresh ETL pipeline for 10-year U.S. Treasury yield data from Alpha Vantage API.

## Architecture
- **Source**: Alpha Vantage Financial API (CSV format)
- **Target**: Delta table `thekitchen.av.ten_year_treasury`
- **Pattern**: Full refresh (overwrite mode)
- **Compute**: Databricks Serverless (Python + PySpark)

## Workflow
1. Configure API parameters (10-year maturity)
2. Fetch complete historical dataset
3. Transform and standardize
4. Overwrite Delta table
5. Verify results

In [0]:
# ==============================================================================
# Alpha Vantage API Documentation
# ==============================================================================
# Alpha Vantage Parameters

# ❚ Required: function
# The function of your choice. In this case, function=TREASURY_YIELD

# ❚ Optional: interval
# By default, interval=monthly. Strings daily, weekly, and monthly are accepted.

# ❚ Optional: maturity
# By default, maturity=10year. Strings 3month, 2year, 5year, 7year, 10year, and 30year are accepted.

# ❚ Optional: datatype
# By default, datatype=json. Strings json and csv are accepted with the following specifications:
# json returns the time series in JSON format;
# csv returns the time series as a CSV (comma separated value) file.

In [0]:
import requests


# API key for Alpha Vantage
api_key = ""

# Define the base URL and parameters
function = "TREASURY_YIELD"
interval = "daily"
maturity = "10year"  # 10-year maturity (benchmark rate)
datatype = "csv"

In [0]:
# Construct the URL
url = f"https://www.alphavantage.co/query?function={function}&interval={interval}&maturity={maturity}&datatype={datatype}&apikey={api_key}"

# Make the API request
r = requests.get(url)
data = r.text

# Print the data
print("\n".join(data.splitlines()[:5]))

In [0]:
import pandas as pd
import io
from pyspark.sql import functions as F

# Read CSV
df = pd.read_csv(io.StringIO(data))

# Replace "." with None in 'value' column
df["value"] = df["value"].replace(".", None)

# Convert 'timestamp' column to datetime
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce").dt.date

# Convert 'value' column to float, coerce errors to NaN
df["value"] = pd.to_numeric(df["value"], errors="coerce")

# Pandas → Spark
spark_df = spark.createDataFrame(df)

# Add audit columns in Spark
spark_df = spark_df.withColumn(
    "CreatedAt",
    F.date_trunc(
        "second", F.from_utc_timestamp(F.current_timestamp(), "Europe/Berlin")
    ),
).withColumn("UpdatedAt", F.lit(None).cast("timestamp"))
table_name = "thekitchen.av.ten_year_treasury"

try:
    (
        spark_df.write.format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable(table_name)
    )
    print(f"Table '{table_name}' written successfully.")
except Exception as e:
    print(f"Failed to write table '{table_name}': {e}")

---

## Technical Highlights

### Design Pattern
* **Full Refresh**: Overwrites entire table with complete dataset
* **Benchmark Rate**: 10-year yields are the most widely watched Treasury rate
* **Audit Trail**: CreatedAt column for data lineage

### Why 10-Year Treasury?
* **Economic Benchmark**: Most referenced Treasury maturity globally
* **Mortgage Rates**: Direct influence on mortgage and loan pricing
* **Risk-Free Rate**: Foundation for equity valuation models (CAPM)
* **Yield Curve**: Central point for yield curve analysis

### Key Metrics
* **Pattern**: Full table overwrite
* **Historical Coverage**: Complete 10-year treasury yield history
* **Use Case**: Economic analysis, risk-free rate modeling, yield curve construction

In [0]:
display(spark_df.orderBy("timestamp", ascending=False).limit(10))